<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z338_GalactusRegLineal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Galactus sobre Regresión Lineal

## La idea

La reg lineal sobre los últimos N meses calcula una pendiente y extrapola.
Pero, ¿esa pendiente es señal real o ruido?

El **experimento Galactus** lo testea directo:
- Tomás la serie de cada producto
- Shuffleás los valores dentro de la ventana (destruís el orden temporal)
- Aplicás la misma reg lineal
- Comparás el RMSE real vs RMSE shuffleado

```
Si RMSE_shuffle ≈ RMSE_real  →  la tendencia es ilusoria, la recta ajusta ruido
Si RMSE_shuffle >> RMSE_real →  había señal real en el orden temporal
```

Lo hacemos por producto y también globalmente, con N repeticiones del shuffle
para tener una distribución del RMSE bajo la hipótesis nula (sin orden).

## Variantes
Barremos ventanas de 3, 6, 12 meses para ver en cuál la señal es más real.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'competencia':    'labo-iii-2026-rosario',
    'periodo_corte':  201910,
    'periodo_target': 201912,
    'n_shuffles':     200,    # repeticiones por producto
    'semilla':        42,
    'ventanas':       [3, 6, 12],
    'horizonte':      2,
}

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
tb_train     = tb_ventas.filter(pl.col('periodo') <= PARAM['periodo_corte'])
tb_real      = (tb_ventas.filter(pl.col('periodo') == PARAM['periodo_target'])
                .select(['product_id','tn']).rename({'tn':'tn_real'}))
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

# Funciones

In [ ]:
def reg_lineal(serie, ventana, horizonte=2):
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)


def reg_lineal_shuffle(serie, ventana, horizonte=2, rng=None):
    """Igual que reg_lineal pero shufflea los valores dentro de la ventana."""
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:].copy()
    if rng is not None:
        rng.shuffle(y)
    else:
        np.random.shuffle(y)
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)

print('OK')

# Experimento Galactus — global

Para cada ventana: RMSE real vs distribución de RMSE bajo shuffle (200 repeticiones).
Si el RMSE real cae en la cola izquierda de la distribución shuffle → hay señal.

In [ ]:
rng = np.random.default_rng(PARAM['semilla'])
N   = PARAM['n_shuffles']
h   = PARAM['horizonte']

# precalcular series
series_train = {
    pid: tb_train.filter(pl.col('product_id') == pid)
         .sort('periodo')['tn'].to_numpy().astype(float)
    for pid in productos
}
reales = {row['product_id']: row['tn_real'] for row in tb_real.to_dicts()}

resultados_galactus = {}

for ventana in PARAM['ventanas']:
    # RMSE real
    errores_real = []
    for pid in productos:
        s    = series_train[pid]
        pred = reg_lineal(s, ventana, h)
        errores_real.append((pred - reales[pid]) ** 2)
    rmse_real = float(np.sqrt(np.mean(errores_real)))

    # distribución RMSE shuffle
    rmse_shuffles = []
    for _ in range(N):
        errores_sh = []
        for pid in productos:
            s    = series_train[pid]
            pred = reg_lineal_shuffle(s, ventana, h, rng=rng)
            errores_sh.append((pred - reales[pid]) ** 2)
        rmse_shuffles.append(float(np.sqrt(np.mean(errores_sh))))

    rmse_shuffles = np.array(rmse_shuffles)
    pval = float((rmse_shuffles <= rmse_real).mean())  # fracción de shuffles ≤ real

    resultados_galactus[ventana] = {
        'rmse_real':     rmse_real,
        'rmse_sh_mean':  float(rmse_shuffles.mean()),
        'rmse_sh_std':   float(rmse_shuffles.std()),
        'pval':          pval,
        'distribucion':  rmse_shuffles,
    }

    señal = 'HAY SEÑAL ✓' if pval < 0.05 else 'sin señal  ✗'
    print(f"ventana={ventana:2d}m  RMSE_real={rmse_real:.4f}  "
          f"RMSE_shuffle={rmse_shuffles.mean():.4f}±{rmse_shuffles.std():.4f}  "
          f"p={pval:.3f}  {señal}")

# Distribución del RMSE bajo shuffle vs RMSE real

In [ ]:
fig, axes = plt.subplots(1, len(PARAM['ventanas']), figsize=(14, 4))

for i, ventana in enumerate(PARAM['ventanas']):
    r    = resultados_galactus[ventana]
    dist = r['distribucion']
    real = r['rmse_real']
    pval = r['pval']

    ax = axes[i]
    ax.hist(dist, bins=30, color='steelblue', edgecolor='white',
            alpha=0.8, label='RMSE shuffles')
    ax.axvline(real, color='tomato', linewidth=2.5, label=f'RMSE real={real:.2f}')
    ax.axvline(dist.mean(), color='gray', linewidth=1, linestyle='--',
               label=f'media shuffle={dist.mean():.2f}')

    color_titulo = 'darkgreen' if pval < 0.05 else 'tomato'
    ax.set_title(f'ventana={ventana}m  p={pval:.3f}', fontsize=9, color=color_titulo)
    ax.set_xlabel('RMSE')
    ax.legend(fontsize=7)

fig.suptitle('Galactus sobre Reg Lineal\n'
             'RMSE real (rojo) vs distribución bajo shuffle (azul)\n'
             'Si el rojo está a la izquierda → hay señal temporal real', fontsize=9)
plt.tight_layout()
plt.show()

# Galactus por producto

Para cada producto: ¿la reg lineal sobre esa serie tiene señal real o no?

Calculamos el p-valor por producto con N shuffles.
- `p < 0.05` → hay orden temporal real en la ventana → la pendiente tiene sentido
- `p >= 0.05` → la recta ajusta ruido → mejor usar mediana

In [ ]:
N_prod   = 100   # shuffles por producto (menos para no tardar siglos)
ventana_analisis = 6

diag_productos = []

for pid in productos:
    s    = series_train[pid]
    real = reales[pid]

    pred_real = reg_lineal(s, ventana_analisis, h)
    err_real  = abs(pred_real - real)

    # distribución de errores bajo shuffle
    errs_sh = []
    for _ in range(N_prod):
        pred_sh = reg_lineal_shuffle(s, ventana_analisis, h, rng=rng)
        errs_sh.append(abs(pred_sh - real))

    errs_sh = np.array(errs_sh)
    pval    = float((errs_sh <= err_real).mean())

    diag_productos.append({
        'product_id':  pid,
        'err_real':    err_real,
        'err_sh_mean': float(errs_sh.mean()),
        'err_sh_std':  float(errs_sh.std()),
        'pval':        pval,
        'tiene_señal': pval < 0.05,
        'pred_real':   pred_real,
        'tn_real':     real,
    })

tb_diag = pl.DataFrame(diag_productos)

n_señal = tb_diag['tiene_señal'].sum()
print(f"Productos con señal temporal real (p<0.05): {n_señal} de {len(productos)} "
      f"({100*n_señal/len(productos):.1f}%)")
print(f"Productos sin señal (ruido puro):           {len(productos)-n_señal} "
      f"({100*(len(productos)-n_señal)/len(productos):.1f}%)")

print()
print("RMSE reg_lineal en productos CON señal:")
con = tb_diag.filter(pl.col('tiene_señal'))
sin = tb_diag.filter(~pl.col('tiene_señal'))
print(f"  con señal: {float(np.sqrt((con['err_real']**2).mean())):.4f}  (n={con.height})")
print(f"  sin señal: {float(np.sqrt((sin['err_real']**2).mean())):.4f}  (n={sin.height})")

# Modelo adaptativo Galactus

Usamos el p-valor por producto como selector:
- `p < 0.05` → tiene señal → reg lineal
- `p >= 0.05` → ruido → mediana reciente

In [ ]:
pval_dict = {row['product_id']: row['pval'] for row in tb_diag.to_dicts()}

for alpha_sel in [0.05, 0.10, 0.20]:
    errores = []
    n_reg   = 0
    for pid in productos:
        s    = series_train[pid]
        real = reales[pid]
        if pval_dict[pid] < alpha_sel:
            pred = reg_lineal(s, ventana_analisis, h)
            n_reg += 1
        else:
            pred = max(float(np.median(s[-6:])), 0.0)
        errores.append((pred - real) ** 2)
    rmse = float(np.sqrt(np.mean(errores)))
    print(f"alpha={alpha_sel:.2f}  → reg en {n_reg:3d} productos, mediana en {len(productos)-n_reg:3d}  "
          f"RMSE={rmse:.4f}")

# baseline
err_base = []
for pid in productos:
    s = series_train[pid]
    err_base.append((reg_lineal(s, ventana_analisis, h) - reales[pid])**2)
print(f"\nbaseline reg{ventana_analisis}m puro:  RMSE={float(np.sqrt(np.mean(err_base))):.4f}")

err_naive = []
for pid in productos:
    s = series_train[pid]
    err_naive.append((max(float(np.median(s[-6:])), 0.0) - reales[pid])**2)
print(f"baseline mediana 6m:     RMSE={float(np.sqrt(np.mean(err_naive))):.4f}")

# Visualización — distribución de p-valores por producto

In [ ]:
pvals = tb_diag['pval'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pvals, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(0.05, color='tomato', linestyle='--', linewidth=1.5, label='α=0.05')
axes[0].axvline(0.10, color='orange', linestyle='--', linewidth=1.5, label='α=0.10')
axes[0].set_xlabel('p-valor por producto')
axes[0].set_ylabel('productos')
axes[0].set_title(f'Distribución de p-valores Galactus\n'
                  f'ventana={ventana_analisis}m, {N_prod} shuffles')
axes[0].legend()

# error real vs error shuffle medio por producto
err_real_arr = tb_diag['err_real'].to_numpy()
err_sh_arr   = tb_diag['err_sh_mean'].to_numpy()
colores      = np.where(tb_diag['tiene_señal'].to_numpy(), 'tomato', 'steelblue')

axes[1].scatter(err_sh_arr, err_real_arr, c=colores, s=10, alpha=0.5)
lim = max(err_sh_arr.max(), err_real_arr.max())
axes[1].plot([0, lim], [0, lim], 'k--', linewidth=0.8)
axes[1].set_xlabel('error medio shuffle')
axes[1].set_ylabel('error real')
axes[1].set_title('Error real vs error shuffle por producto\n'
                  'rojo=tiene señal, azul=ruido\n'
                  'debajo de diagonal = modelo gana al shuffle')

plt.tight_layout()
plt.show()

# Submit — modelo adaptativo Galactus

In [ ]:
import shutil

def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

ruta_drive = '/content/buckets/b1/exp/GalactusRegLineal'
os.makedirs(ruta_drive, exist_ok=True)

# recalcular p-valores con toda la historia (hasta 201912)
series_full = {
    pid: tb_ventas.filter(pl.col('product_id') == pid)
         .sort('periodo')['tn'].to_numpy().astype(float)
    for pid in productos
}

for alpha_sel in [0.05, 0.10]:
    preds = []
    n_reg = 0
    for pid in productos:
        s = series_full[pid]
        if pval_dict[pid] < alpha_sel:
            pred = reg_lineal(s, ventana_analisis, h)
            n_reg += 1
        else:
            pred = max(float(np.median(s[-6:])), 0.0)
        preds.append({'product_id': pid, 'tn': pred})

    tb_final = pl.DataFrame(preds)
    archivo  = f'galactus_reg{ventana_analisis}m_a{int(alpha_sel*100)}.csv'
    mensaje  = f'Galactus adaptativo reg{ventana_analisis}m alpha={alpha_sel} n_reg={n_reg}'

    tb_final.write_csv(archivo)
    shutil.copy(archivo, f'{ruta_drive}/{archivo}')
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'submitted: {archivo}  (reg en {n_reg} productos)')